# Mineral Exploration Budget Allocation: A Python Optimisation Project - Data Cleaning Process

## Overview

From "Resouces and energy quarterly: June 2026" in the Department of Industry, Science and Resources, it was forecasted that export earnings for gold can reach $73 billion from 2026-2027. We can simply investigate this claim on a mathematical scale using a simple constrained linear programming (LP) approach. We begin by extracting data sheets from the full report which are the following:
    1. MonthlyPrices.csv: has the monthly resources and energy benchmark prices
    2. AnnualExp.csv: annual private mineral expenditure in Australia

This project aims to clean these two datasets for a seamless LP pipeline (see Project3_Analysis in GitHub). Gold and Iron ore will be the minerals of focus containing expenditure/prices data on a yearly basis from 1990 to 2026. Additionally, the exchange rates from USD to AUD (retrieved from the Reserve Bank of Australia) is essential for MonthlyPrices.csv since prices are in USD; it is critical that currencies on both datasets must match.

Limitations:
    1. The time period of this analysis will only be from 1990 to 2025 due to bounded availability of commodity price data
    2. This analysis is not a precise commodity-specific relationship rather, a total (all-commodity) state expenditure against a single commodity's price as a broad market-condition indicator

## Library imports and Data Cleaning

In [2]:
import pandas as pd
import numpy as np
from IPython.display import display

/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


### Annual Expenditure Data

In [14]:
df1 = pd.read_csv('AnnualExp.csv', skiprows=5)

#Drop first 5 columns
df1=df1.iloc[:,5:]

#Remove 'unit' column (redundant)
df1.drop("Unnamed: 6", axis=1, inplace=True)

#Rename 0,0 value as 'Year'
df1.iloc[0, 0] = "Year"

#drop trailing columns
df1 = df1.dropna(axis=1, how="all")

display(df1)

,"14 Annual private mineral exploration expenditure, Australia",Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,...,Unnamed: 32,Unnamed: 33,Unnamed: 34,Unnamed: 35,Unnamed: 36,Unnamed: 37,Unnamed: 38,Unnamed: 39,Unnamed: 40,Unnamed: 41
0,Year,1989–90,1990–91,1991–92,1992–93,1993–94,1994–95,1995–96,1996–97,1997–98,...,2014–15,2015–16,2016–17,2017–18,2018–19,2019–20,2020–21,2021–22,2022–23,2023–24
1,Energy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Petroleum,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Onshore,143,217,135,115,145,171,175,252,232,...,"1,254",498,427,347,438,678,623,686,593,805
4,Offshore,439,365,339,497,362,502,514,557,686,...,"2,537","1,278",949,681,823,585,376,462,316,461
5,Total,583,582,474,612,507,673,689,809,918,...,"3,791","1,776","1,376","1,028","1,261","1,263","1,000","1,148",909,"1,266"
6,Coal,33,23,28,24,28,38,53,71,65,...,252,173,120,154,182,303,235,226,281,344
7,Uranium,19,13,13,9,8,8,7,13,22,...,41,38,22,13,14,8,10,17,na,58
8,Total,635,618,514,645,542,719,749,892,"1,005",...,"4,084","1,988","1,518","1,195","1,457","1,574","1,244","1,390",na,"1,667"
9,Metals and other minerals,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
#Filter selected minerals only

#Strip whitespace just in case there are hidden spaces
df1['14  Annual private mineral exploration expenditure, Australia'] = df1['14  Annual private mineral exploration expenditure, Australia'].astype(str).str.strip()

#Create list of minerals from appropriate column
minerals = ["Year", "Gold", "Iron ore"]
mask = df1['14  Annual private mineral exploration expenditure, Australia'].isin(minerals)

#Apply filter to original dataframe
new_df1 = df1[mask]

#Transpose to change wide to long data format and make first 
#row as the column names
new_df1 = new_df1.T
new_df1.columns = new_df1.iloc[0]
new_df1 = new_df1[1:] #removes first row so it doesn't duplicate as data

#Convert gold and iron columns to numeric
int_cols = ["Gold", "Iron ore"]
new_df1[int_cols] = new_df1[int_cols].replace(',', '', regex=True)
new_df1[int_cols] = new_df1[int_cols].apply(pd.to_numeric, errors='coerce')

#Change to int 
new_df1[int_cols] = new_df1[int_cols].astype(int)

#Change year to datetime
new_df1['Year'] = new_df1['Year'].str[:4].astype(int)+1

#Download to csv file
display(new_df1)
new_df1.to_csv("expenditure.csv", index=False)

"14 Annual private mineral exploration expenditure, Australia",Year,Gold,Iron ore
Unnamed: 7,1990,341,11
Unnamed: 8,1991,306,11
Unnamed: 9,1992,305,37
Unnamed: 10,1993,320,24
Unnamed: 11,1994,454,19
Unnamed: 12,1995,555,12
Unnamed: 13,1996,547,14
Unnamed: 14,1997,728,26
Unnamed: 15,1998,648,30
Unnamed: 16,1999,486,42


## Exchange Rate Data

In [16]:
#Read files
rates_2009 = pd.read_csv('2009.csv')
rates_2010 = pd.read_csv('2010.csv')

#Join two datasets into one 
rates_combined = pd.concat([rates_2009, rates_2010], ignore_index=True)

#Convert to datetime and include only years >= 1990
rates_combined['Year'] = pd.to_datetime(rates_combined['Date'], format='%d-%b-%y')
rates_combined = rates_combined[rates_combined['Year'].dt.year>=1990]
rates_combined['Financial Year'] = rates_combined['Year'].dt.year.where(rates_combined['Year'].dt.month < 7,rates_combined['Year'].dt.year+1)

#Aggregate prices using average
annual_rates = rates_combined.groupby('Financial Year')['Rate'].mean().reset_index()
display(annual_rates)
annual_rates.to_csv('annual_rates.csv', index=False)


,Financial Year,Rate
0,1990,0.765567
1,1991,0.785100
2,1992,0.769158
3,1993,0.698175
4,1994,0.691892
5,1995,0.740467
6,1996,0.762917
7,1997,0.780642
8,1998,0.677517
9,1999,0.624692


### Commodity Data

In [17]:

#Upload and skip 4 rows since they're empty
df2 = pd.read_csv('MonthlyPrices.csv', skiprows=4)

#Drop first 4 columns until the date column starts
df2=df2.iloc[:,5:]

#drop trailing columns at the end
df2 = df2.dropna(axis=1, how="all")

#Make row 1 new index
df2.columns = df2.iloc[1]

#Drop row from data
df2[2:]

#Reset index
df2.reset_index(drop=True)

#Clear column name leftover
df2.columns.name=None
df2 = df2.drop(index=range(0,7)) #Rows inbetween consists NaN rows
                                 #and only unit and ID rows
df2 = df2.reset_index(drop=True)

display(df2)

,NaN,Aluminium LME cash,Alumina fob Western Australia,Premium hard coking coal fob East Coast Australia,Thermal coal fob Newcastle 6000 kc,Gold price LBMA PM,Iron ore fines fob Australia a,Argus North East Asian LNG spot price b,Oil WTI spot price,Oil Brent spot price,Uranium industry spot price c,Copper LME cash,Lead LME cash,Lithium spodumene China spot price,Lithium lithium hydroxide China spot price,Zinc LME cash,Silver London fix c,Nickel LME cash
0,January 1990,"1,529",na,na,na,410,na,na,na,na,9,"2,358",705,0,0,"1,297",na,"7,068"
1,February 1990,"1,455",na,na,na,417,na,na,na,na,9,"2,358",779,0,0,"1,396",na,"6,988"
2,March 1990,"1,568",na,na,na,396,na,na,na,na,9,"2,629","1,064",0,0,"1,607",na,"9,282"
3,April 1990,"1,526",na,na,na,375,na,na,na,na,9,"2,692",837,0,0,"1,683",na,"8,944"
4,May 1990,"1,528",na,na,na,369,na,na,na,na,9,"2,742",826,0,0,"1,776",na,"8,710"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
557,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
558,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
559,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
560,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
#Rename 0 column as "Year"
df2 = df2.rename(columns={df2.columns[0]: "Year"})

#Keep 1,6,7 columns for year, Gold, and Iron minerals 

data = ["Year", "Gold price LBMA PM", "Iron ore fines fob Australia a"]
new_df2 = df2.loc[:, df2.columns.isin(data)].copy() #cut ties with df2

#Remove trailing rows and last 4 rows (footnotes)
new_df2 = new_df2.dropna(how='all')
new_df2 = new_df2.iloc[:-4]

#Rename columns
new_df2 = new_df2.rename(columns={'Gold price LBMA PM':'Gold prices', 
                                 'Iron ore fines fob Australia a':'Iron ore prices'})
#Convert gold and iron columns to numeric
num_cols = ["Gold prices", "Iron ore prices"]
new_df2[num_cols] = new_df2[num_cols].replace(',', '', regex=True)
new_df2[num_cols] = new_df2[num_cols].apply(pd.to_numeric, errors='coerce')

#Show and download to csv file
display(new_df2)

,Year,Gold prices,Iron ore prices
0,January 1990,410,NaN
1,February 1990,417,NaN
2,March 1990,396,NaN
3,April 1990,375,NaN
4,May 1990,369,NaN
...,...,...,...
418,November 2024,2651,84.0
419,December 2024,2644,91.0
420,January 2025,2710,89.0
421,February 2025,2895,93.0


In [24]:
#Convert Year column 
new_df2['Year'] = pd.to_datetime(new_df2['Year'], format='%B %Y')
new_df2['Financial Year'] = new_df2['Year'].dt.year.where(new_df2['Year'].dt.month < 7, new_df2['Year'].dt.year+1)

#Aggregate prices using average
yearly_prices = new_df2.groupby('Financial Year')[['Gold prices', 'Iron ore prices']].mean().reset_index()

#Merge exchange rate and prices data
merged_prices = yearly_prices.merge(annual_rates, on='Financial Year')
#Convert USD prices to AUD
merged_prices['Gold prices AUD'] = merged_prices['Gold prices'] / merged_prices['Rate']
merged_prices['Iron ore prices AUD'] = merged_prices['Iron ore prices'] / merged_prices['Rate']
final_prices = merged_prices[['Financial Year', 'Gold prices AUD', 'Iron ore prices AUD']]
display(final_prices)

final_prices.to_csv('prices.csv', index=False)

,Financial Year,Gold prices AUD,Iron ore prices AUD
0,1990,505.725606,NaN
1,1991,475.947862,NaN
2,1992,457.643095,NaN
3,1993,492.713145,NaN
4,1994,547.291845,NaN
5,1995,519.042046,NaN
6,1996,510.977608,NaN
7,1997,465.535831,NaN
8,1998,451.649406,NaN
9,1999,457.559062,NaN
